# Modelado Avanzado y Tuning de Hiperparámetros — Scoring de Intención de Compra

**Objetivo:** extender la comparativa de modelos de `modeling_mvp.ipynb` (Sprint 1) con
algoritmos ensemble adicionales — LightGBM y CatBoost, sumados a XGBoost — y optimizar sus
hiperparámetros con Optuna, maximizando **PR-AUC (`average_precision`)** en vez de F1: PR-AUC
no depende de un umbral de decisión, mientras que F1 asume el umbral 0.5 por defecto de
scikit-learn, que no es el que usa el motor de recomendación (`src/recommender.py` decide con
umbrales de negocio 0.7/0.3, no 0.5). Es además el criterio que ya usó el equipo para elegir el
modelo final en Sprint 1 (ROC-AUC/PR-AUC).

`modeling_mvp.ipynb` es un entregable de Sprint 1 ya aprobado y no se modifica — este notebook
es un artefacto nuevo de Sprint 2.

Pasos:

1. Cargar los artefactos de `feature_enginering.ipynb` (mismo split/preprocesador que usó Sprint 1).
2. Reentrenar Random Forest y XGBoost (misma configuración que Sprint 1) como referencia directa.
3. Entrenar LightGBM y CatBoost sin tunear, con el mismo manejo de desbalance por pesos de clase.
4. Tunear XGBoost, LightGBM y CatBoost con Optuna (Stratified 3-Fold CV sobre train, optimizando PR-AUC).
5. Comparar todo bajo las mismas métricas del proyecto y elegir el mejor por PR-AUC.
6. Verificar que el motor de recomendación (`asignar_accion`/`resumen_acciones`) sigue siendo coherente con el modelo ganador.
7. Persistir el nuevo modelo final (y sus hiperparámetros) solo si supera al Random Forest actual.

## 1. Configuración y carga de artefactos

In [1]:
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import optuna
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import comparar_modelos, evaluar_modelo
from src.recommender import asignar_accion, resumen_acciones

optuna.logging.set_verbosity(optuna.logging.WARNING)

CARPETA_MODELOS = PROJECT_ROOT / "data" / "models"
split = joblib.load(CARPETA_MODELOS / "train_test_split.joblib")

X_train, X_test = split["X_train"], split["X_test"]
y_train, y_test = split["y_train"], split["y_test"]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}, balance: {y_train.mean():.4f}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}, balance: {y_test.mean():.4f}")

/home/juanma/HENRRY/PROYECTO_FINAL/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X_train: (9764, 78), y_train: (9764,), balance: 0.1563
X_test: (2441, 78), y_test: (2441,), balance: 0.1565


`X_train`/`X_test` son las mismas matrices ya codificadas y escaladas que usó
`modeling_mvp.ipynb` (`preprocessor`, ajustado solo con train en `feature_enginering.ipynb`).

## 2. Referencia Sprint 1: Random Forest y XGBoost

Se reentrenan con exactamente la misma configuración que en `modeling_mvp.ipynb`, para
tener el punto de comparación directo en la misma tabla que los modelos nuevos.

In [2]:
resultados = {}
modelos = {}

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight (razón negativos/positivos en train): {scale_pos_weight:.4f}")

rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
resultados["random_forest"] = evaluar_modelo(
    y_test, rf.predict(X_test), rf.predict_proba(X_test)[:, 1]
)
modelos["random_forest"] = rf
resultados["random_forest"]

scale_pos_weight (razón negativos/positivos en train): 5.3984


{'precision': 0.6577669902912622,
 'recall': 0.7094240837696335,
 'f1': 0.6826196473551638,
 'roc_auc': 0.924264943333952,
 'average_precision': 0.7226931082367187}

In [3]:
xgb_base = XGBClassifier(
    n_estimators=300,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
)
xgb_base.fit(X_train, y_train)
resultados["xgboost"] = evaluar_modelo(
    y_test, xgb_base.predict(X_test), xgb_base.predict_proba(X_test)[:, 1]
)
modelos["xgboost"] = xgb_base
resultados["xgboost"]

{'precision': 0.6458852867830424,
 'recall': 0.6780104712041884,
 'f1': 0.6615581098339719,
 'roc_auc': 0.9220641596464507,
 'average_precision': 0.7170867562321463}

## 3. Modelos ensemble adicionales (baseline, sin tuning)

Mismo criterio de manejo del desbalance (~85/15) que ya usa el proyecto, adaptado a cada
librería: `class_weight="balanced"` en LightGBM (equivalente a scikit-learn) y
`auto_class_weights="Balanced"` en CatBoost (su forma nativa de ponderar la clase minoritaria).

In [4]:
lgbm_base = LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)
lgbm_base.fit(X_train, y_train)
resultados["lightgbm"] = evaluar_modelo(
    y_test, lgbm_base.predict(X_test), lgbm_base.predict_proba(X_test)[:, 1]
)
modelos["lightgbm"] = lgbm_base
resultados["lightgbm"]

{'precision': 0.5722326454033771,
 'recall': 0.7984293193717278,
 'f1': 0.6666666666666666,
 'roc_auc': 0.9318697888722477,
 'average_precision': 0.7474171703056453}

In [5]:
cat_base = CatBoostClassifier(auto_class_weights="Balanced", random_state=42, verbose=False)
cat_base.fit(X_train, y_train)
resultados["catboost"] = evaluar_modelo(
    y_test, cat_base.predict(X_test), cat_base.predict_proba(X_test)[:, 1]
)
modelos["catboost"] = cat_base
resultados["catboost"]

{'precision': 0.5733333333333334,
 'recall': 0.7879581151832461,
 'f1': 0.6637265711135611,
 'roc_auc': 0.9317464636165067,
 'average_precision': 0.7379321911450423}

In [6]:
print("Comparativa baseline (antes de tuning), ordenada por PR-AUC:")
comparar_modelos(resultados, criterio="average_precision")

Comparativa baseline (antes de tuning), ordenada por PR-AUC:


,precision,recall,f1,roc_auc,average_precision
lightgbm,0.572233,0.798429,0.666667,0.931870,0.747417
catboost,0.573333,0.787958,0.663727,0.931746,0.737932
random_forest,0.657767,0.709424,0.682620,0.924265,0.722693
xgboost,0.645885,0.678010,0.661558,0.922064,0.717087


## 4. Tuning de hiperparámetros con Optuna

Un `study` de Optuna por modelo (XGBoost, LightGBM, CatBoost), maximizando el promedio de
`average_precision` (PR-AUC) en Stratified 3-Fold CV **sobre train únicamente** — el test set
quedafuera de la búsqueda y solo se usa al final para evaluar el modelo ya elegido, igual que en
el resto del proyecto. Cada estudio corre hasta 30 trials o 4 minutos (lo que ocurra primero),
para mantener el tiempo total acotado. El manejo del desbalance de clases (`scale_pos_weight` /
`class_weight="balanced"` / `auto_class_weights="Balanced"`) se deja fijo en cada modelo — igual
que en la sección anterior — y Optuna solo busca sobre los hiperparámetros de complejidad/regularización
del árbol.

In [7]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
N_TRIALS = 30
TIMEOUT_S = 240


def cv_average_precision(model_cls, params):
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        modelo = model_cls(**params)
        y_fit = y_train.iloc[train_idx] if hasattr(y_train, "iloc") else y_train[train_idx]
        modelo.fit(X_train[train_idx], y_fit)
        y_val = y_train.iloc[val_idx] if hasattr(y_train, "iloc") else y_train[val_idx]
        prob = modelo.predict_proba(X_train[val_idx])[:, 1]
        scores.append(average_precision_score(y_val, prob))
    return float(np.mean(scores))

In [8]:
def objective_xgb(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    )
    return cv_average_precision(XGBClassifier, params)


def objective_lgbm(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        class_weight="balanced",
        random_state=42,
        verbose=-1,
        n_jobs=-1,
    )
    return cv_average_precision(LGBMClassifier, params)


def objective_cat(trial):
    params = dict(
        iterations=trial.suggest_int("iterations", 100, 500),
        depth=trial.suggest_int("depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        auto_class_weights="Balanced",
        random_state=42,
        verbose=False,
        thread_count=-1,
    )
    return cv_average_precision(CatBoostClassifier, params)

In [9]:
tuning_specs = {
    "xgboost_tuned": (
        objective_xgb, XGBClassifier,
        {"scale_pos_weight": scale_pos_weight, "random_state": 42, "eval_metric": "logloss", "n_jobs": -1},
    ),
    "lightgbm_tuned": (
        objective_lgbm, LGBMClassifier,
        {"class_weight": "balanced", "random_state": 42, "verbose": -1, "n_jobs": -1},
    ),
    "catboost_tuned": (
        objective_cat, CatBoostClassifier,
        {"auto_class_weights": "Balanced", "random_state": 42, "verbose": False, "thread_count": -1},
    ),
}

mejores_hiperparametros = {}
estudios = {}

for nombre, (objective, model_cls, params_fijos) in tuning_specs.items():
    t0 = time.time()
    study = optuna.create_study(direction="maximize", study_name=nombre)
    study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_S, show_progress_bar=False)
    estudios[nombre] = study

    mejores_params = {**study.best_params, **params_fijos}
    modelo_tuneado = model_cls(**mejores_params)
    modelo_tuneado.fit(X_train, y_train)

    resultados[nombre] = evaluar_modelo(
        y_test, modelo_tuneado.predict(X_test), modelo_tuneado.predict_proba(X_test)[:, 1]
    )
    modelos[nombre] = modelo_tuneado
    mejores_hiperparametros[nombre] = study.best_params

    print(
        f"{nombre}: {time.time()-t0:.1f}s, {len(study.trials)} trials, "
        f"mejor PR-AUC en CV={study.best_value:.4f}, PR-AUC en test={resultados[nombre]['average_precision']:.4f}"
    )

xgboost_tuned: 31.3s, 30 trials, mejor PR-AUC en CV=0.7507, PR-AUC en test=0.7525


lightgbm_tuned: 22.6s, 30 trials, mejor PR-AUC en CV=0.7523, PR-AUC en test=0.7577


catboost_tuned: 205.1s, 30 trials, mejor PR-AUC en CV=0.7542, PR-AUC en test=0.7514


## 5. Comparativa final

In [10]:
tabla_comparativa = comparar_modelos(resultados, criterio="average_precision")
tabla_comparativa

,precision,recall,f1,roc_auc,average_precision
lightgbm_tuned,0.540230,0.861257,0.663976,0.938520,0.757678
xgboost_tuned,0.548986,0.850785,0.667351,0.937961,0.752504
catboost_tuned,0.558669,0.835079,0.669465,0.935833,0.751441
lightgbm,0.572233,0.798429,0.666667,0.931870,0.747417
catboost,0.573333,0.787958,0.663727,0.931746,0.737932
random_forest,0.657767,0.709424,0.682620,0.924265,0.722693
xgboost,0.645885,0.678010,0.661558,0.922064,0.717087


In [11]:
mejor_modelo_nombre = tabla_comparativa.index[0]
mejor_modelo = modelos[mejor_modelo_nombre]
print(f"Mejor modelo por PR-AUC: {mejor_modelo_nombre}")
print(tabla_comparativa.loc[mejor_modelo_nombre])

Mejor modelo por PR-AUC: lightgbm_tuned
precision            0.540230
recall               0.861257
f1                   0.663976
roc_auc              0.938520
average_precision    0.757678
Name: lightgbm_tuned, dtype: float64


Se elige por `average_precision` (PR-AUC) — más informativo que ROC-AUC cuando la clase
positiva es minoritaria, y coherente con que el motor de recomendación decide sobre dos umbrales
de probabilidad (0.7/0.3), no sobre una única frontera 0.5 como asumiría optimizar por F1.

## 6. Verificación de negocio: motor de recomendación

Igual que en `modeling_mvp.ipynb`: se corrobora que, con el modelo ganador de esta ronda,
`cross_selling` sigue concentrando una tasa de compra real claramente mayor que `retencion`
antes de reemplazar el modelo que sirve el motor de recomendación.

In [12]:
y_prob_final = mejor_modelo.predict_proba(X_test)[:, 1]
acciones = asignar_accion(y_prob_final)
acciones.value_counts()

accion
retencion        1550
sin_accion        457
cross_selling     434
Name: count, dtype: int64

In [13]:
resumen_acciones(acciones, y_test)

,n_sesiones,tasa_compra_real,pct_del_total
accion,,,
retencion,1550,0.011613,63.50
sin_accion,457,0.179431,18.72
cross_selling,434,0.649770,17.78


## 7. Persistencia del modelo final

Solo se reemplaza `data/models/modelo_final.joblib` (el que sirve el motor de recomendación)
si el mejor candidato de esta ronda supera en PR-AUC al Random Forest de Sprint 1. Si ningún
candidato lo supera, se deja constancia explícita y el Random Forest se mantiene como modelo
final.

In [14]:
RUTA_MODELO_FINAL = CARPETA_MODELOS / "modelo_final.joblib"
RUTA_HIPERPARAMETROS = CARPETA_MODELOS / "mejores_hiperparametros.json"

pr_auc_actual = resultados["random_forest"]["average_precision"]
pr_auc_candidato = resultados[mejor_modelo_nombre]["average_precision"]

print(f"PR-AUC modelo final actual (Random Forest, Sprint 1): {pr_auc_actual:.4f}")
print(f"PR-AUC mejor candidato de esta ronda ({mejor_modelo_nombre}): {pr_auc_candidato:.4f}")

if mejor_modelo_nombre != "random_forest" and pr_auc_candidato > pr_auc_actual:
    joblib.dump(mejor_modelo, RUTA_MODELO_FINAL)
    print(f"\nNuevo modelo final: {mejor_modelo_nombre} (guardado en {RUTA_MODELO_FINAL})")

    if mejor_modelo_nombre in mejores_hiperparametros:
        import json

        payload = {
            "modelo": mejor_modelo_nombre,
            "metricas_test": resultados[mejor_modelo_nombre],
            "hiperparametros": mejores_hiperparametros[mejor_modelo_nombre],
        }
        with open(RUTA_HIPERPARAMETROS, "w") as f:
            json.dump(payload, f, indent=2, default=float)
        print(f"Hiperparámetros ganadores guardados en {RUTA_HIPERPARAMETROS}")
else:
    print("\nNingún candidato superó al Random Forest de Sprint 1: se mantiene como modelo final.")

PR-AUC modelo final actual (Random Forest, Sprint 1): 0.7227
PR-AUC mejor candidato de esta ronda (lightgbm_tuned): 0.7577

Nuevo modelo final: lightgbm_tuned (guardado en /home/juanma/HENRRY/PROYECTO_FINAL/Proyecto-Final-Henry/data/models/modelo_final.joblib)
Hiperparámetros ganadores guardados en /home/juanma/HENRRY/PROYECTO_FINAL/Proyecto-Final-Henry/data/models/mejores_hiperparametros.json


## 8. Conclusiones

- LightGBM y CatBoost se suman a la comparativa de Sprint 1 (Random Forest, XGBoost) bajo el
  mismo manejo de desbalance por pesos de clase, sin tuning.
- El tuning con Optuna busca hiperparámetros de complejidad/regularización por Stratified
  3-Fold CV sobre train, optimizando PR-AUC — sin tocar el test set hasta la evaluación final.
- El modelo final se decide por PR-AUC (no F1), porque el motor de recomendación actúa sobre dos
  umbrales de probabilidad (0.7/0.3) y no sobre una única frontera 0.5.
- `modelo_final.joblib` solo se reemplaza si el ganador de esta ronda supera al Random Forest de
  Sprint 1; de lo contrario, el resultado queda documentado igual, como evidencia de que se
  probaron alternativas más sofisticadas.
- Pendiente para más adelante (fuera de este notebook): Precision@K/Recall@K sobre el ranking,
  demo (API + Streamlit), dashboard, pipeline reproducible con MLflow/CI y documentación
  (ADRs, plan de validación, manual de usuario).